# Clustering: K-Means without PCA vs with PCA

## Imports

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
warnings.filterwarnings("ignore")

## Config

In [ ]:
DATA = "clustering_data.csv"
ID_COL = "Record_ID"
SEED = 42
ALPHA = 0.05

## Load Data

In [ ]:
df = pd.read_csv(DATA)
df.head()

## Shape

In [ ]:
print(df.shape)

## Data Types

In [ ]:
df.info()

## Missing Values

In [ ]:
miss = df.isna().sum()
miss[miss > 0]

## Duplicate Rows

In [ ]:
print(df.duplicated().sum())

## Descriptive Statistics

In [ ]:
df.describe().T

## Identify Identifier Column

In [ ]:
df.nunique().sort_values(ascending=False).head(3)

## Feature Types

In [ ]:
X_cols = df.drop(columns=ID_COL)
num_cols = X_cols.select_dtypes(include="number").columns.tolist()
cat_cols = X_cols.select_dtypes(exclude="number").columns.tolist()
print("numeric:", num_cols)
print("categorical:", cat_cols)

## Skewness

In [ ]:
df[num_cols].skew().sort_values()

## Predictor Histograms

In [ ]:
df[num_cols].hist(bins=30, figsize=(14, 9))
plt.tight_layout()
plt.show()

## Predictor QQ Plots

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for col, ax in zip(num_cols, axes.ravel()):
    stats.probplot(df[col].dropna(), plot=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Categorical Counts

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(10, 3.5))
for col, ax in zip(cat_cols, np.atleast_1d(axes)):
    sns.countplot(x=df[col], ax=ax)
plt.show()

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

## Highly Correlated Pairs

In [ ]:
corr = df[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack()
pairs[pairs > 0.8].sort_values(ascending=False)

## Outlier Boxplots

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for col, ax in zip(num_cols, axes.ravel()):
    sns.boxplot(y=df[col], ax=ax)
plt.tight_layout()
plt.show()

## Outlier Count (IQR Rule)

In [ ]:
Q1, Q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
IQR = Q3 - Q1
outliers = (df[num_cols] < Q1 - 1.5 * IQR) | (df[num_cols] > Q3 + 1.5 * IQR)
outliers.sum()

## Drop Identifier

In [ ]:
df = df.drop(columns=ID_COL)

## Impute Categorical (fillna)

In [ ]:
df[cat_cols] = df[cat_cols].fillna("Unknown")
df[cat_cols].isna().sum()

## Encode Categorical

In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
X_raw = df.copy()
df.shape

## Impute Numeric (fillna with median)

In [ ]:
df = df.fillna(df[num_cols].median())
df.isna().sum().sum()

## Outlier Treatment (IQR clip)

In [ ]:
Q1, Q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
IQR = Q3 - Q1
df[num_cols] = df[num_cols].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR, axis=1)

## Scale Features

In [ ]:
X = df.copy()
X_s = StandardScaler().fit_transform(X)

## Choose K: Metrics for Each K

In [ ]:
rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_s)
    rows.append({"K": k, "Inertia": km.inertia_,
                 "Silhouette": silhouette_score(X_s, km.labels_),
                 "Calinski-Harabasz": calinski_harabasz_score(X_s, km.labels_),
                 "Davies-Bouldin": davies_bouldin_score(X_s, km.labels_)})
k_table = pd.DataFrame(rows).set_index("K")
k_table.round(3)

## Choose K: Elbow and Silhouette Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(k_table.index, k_table["Inertia"], marker="o"); ax[0].set_title("Elbow (inertia)")
ax[1].plot(k_table.index, k_table["Silhouette"], marker="o"); ax[1].set_title("Silhouette")
plt.show()

## Select K

In [ ]:
K = int(k_table["Silhouette"].idxmax())   # or set K = 3 by hand after the elbow plot
print("K chosen (best silhouette):", K)

## K-Means Without PCA

In [ ]:
km1 = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X_s)
labels1 = km1.labels_

## Cluster Sizes Without PCA

In [ ]:
pd.Series(labels1).value_counts().sort_index()

## PCA: Components for 95% Variance

In [ ]:
pca_full = PCA().fit(X_s)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_comp = int(np.argmax(cumvar >= 0.95) + 1)
print("components for 95%:", n_comp)

## PCA: Explained Variance Plot

In [ ]:
plt.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
plt.axhline(0.95, color="r", linestyle="--")
plt.axvline(n_comp, color="g", linestyle="--")
plt.xlabel("Components")
plt.ylabel("Cumulative explained variance")
plt.show()

## Apply PCA

In [ ]:
X_pca = PCA(n_components=n_comp).fit_transform(X_s)
print("original dims:", X_s.shape[1], "| reduced dims:", X_pca.shape[1])

## K-Means With PCA

In [ ]:
km2 = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X_pca)
labels2 = km2.labels_

## Compare Without PCA vs With PCA

In [ ]:
def scores(data, labels):
    return {"Silhouette": silhouette_score(data, labels),
            "Calinski-Harabasz": calinski_harabasz_score(data, labels),
            "Davies-Bouldin": davies_bouldin_score(data, labels)}

pd.DataFrame({"Without PCA": scores(X_s, labels1), "With PCA": scores(X_pca, labels2)}).round(4)

## Agreement Between the Two Clusterings (ARI)

In [ ]:
print(adjusted_rand_score(labels1, labels2))

## Cluster Visualisation (first 2 principal components)

In [ ]:
p2 = PCA(n_components=2).fit_transform(X_s)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(p2[:, 0], p2[:, 1], c=labels1, cmap="viridis", s=12); ax[0].set_title("Without PCA labels")
ax[1].scatter(p2[:, 0], p2[:, 1], c=labels2, cmap="viridis", s=12); ax[1].set_title("With PCA labels")
plt.show()

## Cluster Profiles (original scale means)

In [ ]:
X.groupby(labels1).mean().round(2).T

## Pipeline Version

In [ ]:
pipe1 = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("kmeans", KMeans(n_clusters=K, n_init=10, random_state=SEED))])
pipe2 = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("pca", PCA(n_components=0.95)),
                  ("kmeans", KMeans(n_clusters=K, n_init=10, random_state=SEED))])
pipe1.fit(X_raw)
pipe2.fit(X_raw)

## Pipeline Silhouette

In [ ]:
print("without PCA:", silhouette_score(pipe1[:-1].transform(X_raw), pipe1["kmeans"].labels_))
print("with PCA   :", silhouette_score(pipe2[:-1].transform(X_raw), pipe2["kmeans"].labels_))